# Reto V

In [22]:
!pip install -q langchain langchain-community faiss-cpu duckdb pandas openai tiktoken

In [23]:
# Data
DATASET_PATH = "../data/sql_dataset_bourbaki.json"

In [24]:
import json
from pprint import pprint

In [25]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print("type(dataset):", type(dataset))

if isinstance(dataset, dict):
    print("keys:", list(dataset.keys()))
    first_key = list(dataset.keys())[0]
    print("\nprimer key:", first_key)
    print("type(dataset[first_key]):", type(dataset[first_key]))
    print("\nvalor de ejemplo:")
    pprint(dataset[first_key])

elif isinstance(dataset, list):
    print("len(dataset):", len(dataset))
    print("type(dataset[0]):", type(dataset[0]))
    print("\nprimer elemento:")
    pprint(dataset[0])

else:
    print(dataset)

type(dataset): <class 'dict'>
keys: ['users', 'categories', 'products', 'orders', 'order_items', 'reviews', 'shipping_details', 'promotions']

primer key: users
type(dataset[first_key]): <class 'dict'>

valor de ejemplo:
{'description': 'Almacena información de cuentas de clientes, incluyendo '
                'detalles de contacto y fecha de registro.',
 'examples': [{'description': 'Buscar un usuario por correo electrónico.',
               'sql': 'SELECT * FROM users WHERE email = '
                      "'cliente@example.com';"},
              {'description': 'Contar nuevos usuarios en el último mes.',
               'sql': 'SELECT COUNT(*) FROM users WHERE created_at >= '
                      "CURRENT_DATE - INTERVAL '1 month';"},
              {'description': 'Buscar usuarios sin número de teléfono.',
               'sql': 'SELECT id, first_name, last_name FROM users WHERE phone '
                      'IS NULL;'}],
 'schema': 'id INT PRIMARY KEY, first_name VARCHAR(50), last_na

In [26]:
# Creando documentos
from langchain_core.documents import Document

documents = []
table_lookup = {}

for table_name, item in dataset.items():
    description = item.get("description", "").strip()
    schema = item.get("schema", "").strip()
    examples = item.get("examples", [])

    example_blocks = []
    for i, ex in enumerate(examples, start=1):
        ex_desc = ex.get("description", "").strip()
        ex_sql = ex.get("sql", "").strip()
        example_blocks.append(
            f"Ejemplo {i}:\n"
            f"Descripción: {ex_desc}\n"
            f"SQL: {ex_sql}"
        )

    page_content = (
        f"Tabla: {table_name}\n"
        f"Descripción: {description}\n"
        f"Schema: {schema}\n\n"
        f"Ejemplos:\n" + "\n\n".join(example_blocks)
    )

    doc = Document(
        page_content=page_content,
        metadata={
            "table_name": table_name,
            "source_type": "table_doc"
        }
    )

    documents.append(doc)

    table_lookup[table_name] = {
        "table_name": table_name,
        "description": description,
        "schema": schema,
        "examples": examples
    }

print(f"Documents creados: {len(documents)}")
print("\nMetadata de ejemplo:")
print(documents[0].metadata)
print("\nContenido resumido del primer document:")
print(documents[0].page_content[:800])

Documents creados: 8

Metadata de ejemplo:
{'table_name': 'users', 'source_type': 'table_doc'}

Contenido resumido del primer document:
Tabla: users
Descripción: Almacena información de cuentas de clientes, incluyendo detalles de contacto y fecha de registro.
Schema: id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP

Ejemplos:
Ejemplo 1:
Descripción: Buscar un usuario por correo electrónico.
SQL: SELECT * FROM users WHERE email = 'cliente@example.com';

Ejemplo 2:
Descripción: Contar nuevos usuarios en el último mes.
SQL: SELECT COUNT(*) FROM users WHERE created_at >= CURRENT_DATE - INTERVAL '1 month';

Ejemplo 3:
Descripción: Buscar usuarios sin número de teléfono.
SQL: SELECT id, first_name, last_name FROM users WHERE phone IS NULL;


In [27]:
!pip install -q langchain-openai langchain-community faiss-cpu

In [29]:
## OpenAI API Key
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Ingresa tu OPENAI_API_KEY: ")

In [30]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(documents, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("FAISS listo")

FAISS listo


In [31]:
query = "¿Cuántos pedidos cancelados hubo en 2024?"
results = retriever.invoke(query)

for i, doc in enumerate(results, start=1):
    print(f"\nResultado {i}")
    print(doc.metadata)
    print(doc.page_content[:500])


Resultado 1
{'table_name': 'orders', 'source_type': 'table_doc'}
Tabla: orders
Descripción: Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.
Schema: id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2)

Ejemplos:
Ejemplo 1:
Descripción: Obtener pedidos pendientes para un usuario específico.
SQL: SELECT id, order_date, total_amount FROM orders WHERE user_id = 123 AND status = 'pending';

Ejemplo 2:
Descripción: Calcular los ingresos totales de hoy.
SQL: SELECT SUM(t

Resultado 2
{'table_name': 'shipping_details', 'source_type': 'table_doc'}
Tabla: shipping_details
Descripción: Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.
Schema: id INT PRIMARY KEY, order_id INT, address_line1 VARCHAR(255), city VARCHAR(100), state VARCHAR(100), postal_code VARCHAR(20), tracking_number VARCHAR(100), carrier VARCHAR(50)

Ejemplos:
Ejemplo 1:
Descripción: Buscar

In [47]:
def retrieve_relevant_tables(question: str, retriever, table_lookup: dict, k: int = 3):
    results = retriever.invoke(question)

    selected_tables = []
    context_blocks = []

    seen = set()

    for doc in results[:k]:
        table_name = doc.metadata.get("table_name")

        if not table_name or table_name in seen:
            continue

        seen.add(table_name)

        table_info = table_lookup.get(table_name)
        if not table_info:
            continue

        selected_tables.append(table_name)

        examples = table_info.get("examples", [])
        example_text = []

        for i, ex in enumerate(examples, start=1):
            ex_desc = ex.get("description", "").strip()
            ex_sql = ex.get("sql", "").strip()
            example_text.append(
                f"Ejemplo {i}:\n"
                f"- Descripción: {ex_desc}\n"
                f"- SQL: {ex_sql}"
            )

        block = (
            f"Tabla: {table_name}\n"
            f"Descripción: {table_info.get('description', '')}\n"
            f"Schema: {table_info.get('schema', '')}\n"
            f"Ejemplos:\n" + "\n".join(example_text)
        )

        context_blocks.append(block)

    final_context = "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(context_blocks)

    return {
        "question": question,
        "selected_tables": selected_tables,
        "context": final_context
    }

In [33]:
question = "¿Cuántos pedidos cancelados hubo en 2024?"

retrieval_output = retrieve_relevant_tables(
    question=question,
    retriever=retriever,
    table_lookup=table_lookup,
    k=3
)

print("Tablas seleccionadas:")
print(retrieval_output["selected_tables"])

print("\nContexto:")
print(retrieval_output["context"][:2500])

Tablas seleccionadas:
['orders', 'shipping_details', 'order_items']

Contexto:


Tabla: orders
Descripción: Datos transaccionales de alto nivel para compras de clientes, estado del pedido y costo total.
Schema: id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2)
Ejemplos:
Ejemplo 1:
- Descripción: Obtener pedidos pendientes para un usuario específico.
- SQL: SELECT id, order_date, total_amount FROM orders WHERE user_id = 123 AND status = 'pending';
Ejemplo 2:
- Descripción: Calcular los ingresos totales de hoy.
- SQL: SELECT SUM(total_amount) AS revenue FROM orders WHERE DATE(order_date) = CURRENT_DATE AND status != 'cancelled';
Ejemplo 3:
- Descripción: Contar pedidos cancelados en 2024.
- SQL: SELECT COUNT(*) FROM orders WHERE status = 'cancelled' AND EXTRACT(YEAR FROM order_date) = 2024;


Tabla: shipping_details
Descripción: Información logística, números de seguimiento y direcciones de entrega vinculadas a un pedido.
Schema: id IN

In [54]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def generate_sql(question: str, retrieved_context: str) -> str:
    prompt = f"""
Eres un experto en SQL para DuckDB.

Tu tarea es generar UNA consulta SQL válida y correcta para responder la pregunta del usuario.

Reglas:
- Usa solo las tablas y columnas disponibles en el contexto.
- No inventes tablas ni columnas.
- Genera SQL compatible con DuckDB.
- Devuelve únicamente la consulta SQL.
- No uses markdown.
- No agregues explicaciones.
- Cuando uses agregaciones, asigna aliases claros y legibles.
  Ejemplos:
  - COUNT(*) AS total
  - SUM(total_amount) AS total_revenue
  - AVG(rating) AS average_rating

Pregunta del usuario:
{question}

Contexto disponible:
{retrieved_context}
"""
    response = llm.invoke(prompt)
    return response.content.strip()

In [48]:
def clean_sql(sql_text: str) -> str:
    sql_text = sql_text.strip()

    if sql_text.startswith("```sql"):
        sql_text = sql_text[len("```sql"):].strip()
    elif sql_text.startswith("```"):
        sql_text = sql_text[len("```"):].strip()

    if sql_text.endswith("```"):
        sql_text = sql_text[:-3].strip()

    return sql_text


def answer_question_safe(question: str):
    retrieval_output = retrieve_relevant_tables(
        question=question,
        retriever=retriever,
        table_lookup=table_lookup,
        k=3
    )

    raw_sql = generate_sql(
        question=retrieval_output["question"],
        retrieved_context=retrieval_output["context"]
    )

    sql_query = clean_sql(raw_sql)

    try:
        result_df = conn.execute(sql_query).df()
        error = None
    except Exception as e:
        result_df = None
        error = str(e)

    return {
        "question": question,
        "selected_tables": retrieval_output["selected_tables"],
        "sql_query": sql_query,
        "result_df": result_df,
        "error": error
    }

In [35]:
sql_query = generate_sql(
    question=retrieval_output["question"],
    retrieved_context=retrieval_output["context"]
)

print(sql_query)

SELECT COUNT(*) FROM orders WHERE status = 'cancelled' AND EXTRACT(YEAR FROM order_date) = 2024;


In [36]:
!pip install -q duckdb

In [ ]:
import duckdb

In [40]:
conn = duckdb.connect(database=':memory:')
print("Conexión DuckDB en memoria lista")

Conexión DuckDB en memoria lista


In [41]:
setup_sql = """
-- USERS
CREATE TABLE users (id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, phone VARCHAR(20), created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP);
INSERT INTO users VALUES
(1, 'Ana', 'Gomez', 'cliente@example.com', '555-0101', '2026-02-15 10:00:00'),
(2, 'Carlos', 'Ruiz', 'carlos.r@mail.com', NULL, '2026-03-01 14:30:00'),
(3, 'Beatriz', 'Perez', 'bea.p@mail.com', '555-0103', '2026-03-10 09:15:00'),
(4, 'David', 'Lopes', 'david.l@mail.com', NULL, '2026-01-20 11:20:00'),
(5, 'Elena', 'Torres', 'elena.t@mail.com', '555-0105', '2026-03-16 16:45:00'),
(6, 'Fernando', 'Silva', 'fer.s@mail.com', '555-0106', '2026-02-28 08:00:00'),
(7, 'Gabriela', 'Cruz', 'gabi.c@mail.com', '555-0107', '2026-03-05 13:10:00'),
(8, 'Hugo', 'Diaz', 'hugo.d@mail.com', NULL, '2026-03-12 17:55:00'),
(9, 'Isabel', 'Ortiz', 'isa.o@mail.com', '555-0109', '2026-01-05 12:00:00'),
(10, 'Jorge', 'Rios', 'jorge.r@mail.com', '555-0110', '2026-03-17 09:00:00');

-- CATEGORIES
CREATE TABLE categories (id INT PRIMARY KEY, parent_id INT, name VARCHAR(100), description TEXT);
INSERT INTO categories VALUES
(1, NULL, 'Electronics', 'Gadgets and devices'),
(2, NULL, 'Clothing', 'Apparel and accessories'),
(3, 1, 'Laptops', 'Computers and notebooks'),
(4, 1, 'Smartphones', 'Mobile devices'),
(5, 2, 'Menswear', 'Men clothing'),
(6, 5, 'Shirts', 'T-shirts and button-downs'),
(7, 5, 'Pants', 'Jeans and trousers'),
(8, 2, 'Womenswear', 'Women clothing'),
(9, 8, 'Dresses', 'Casual and formal dresses'),
(10, NULL, 'Home & Garden', 'Furniture and tools');

-- PRODUCTS
CREATE TABLE products (id INT PRIMARY KEY, category_id INT, name VARCHAR(255), price DECIMAL(10, 2), stock_quantity INT, sku VARCHAR(50) UNIQUE, is_active BOOLEAN);
INSERT INTO products VALUES
(1, 3, 'Pro Laptop 15', 1299.99, 15, 'PROD-123', TRUE),
(2, 4, 'Smartphone X', 799.50, 0, 'SKU-002', TRUE),
(3, 6, 'Cotton T-Shirt', 19.99, 100, 'SKU-003', TRUE),
(4, 7, 'Denim Jeans', 49.99, 50, 'SKU-004', TRUE),
(5, 9, 'Summer Dress', 39.99, 0, 'SKU-005', FALSE),
(6, 3, 'Basic Laptop 13', 599.00, 30, 'SKU-006', TRUE),
(7, 10, 'Garden Shovel', 25.00, 20, 'SKU-007', TRUE),
(8, 10, 'Table Lamp', 35.50, 10, 'SKU-008', FALSE),
(9, 4, 'Budget Phone', 299.00, 0, 'SKU-009', TRUE),
(10, 6, 'Polo Shirt', 29.99, 45, 'SKU-010', TRUE);

-- ORDERS
CREATE TABLE orders (id INT PRIMARY KEY, user_id INT, order_date TIMESTAMP, status VARCHAR(50), total_amount DECIMAL(10, 2));
INSERT INTO orders VALUES
(1, 1, '2024-05-10 10:00:00', 'cancelled', 1299.99),
(2, 1, '2026-03-17 08:30:00', 'pending', 49.99),
(3, 2, '2026-03-16 14:00:00', 'completed', 799.50),
(4, 3, '2024-11-20 09:15:00', 'cancelled', 39.99),
(5, 4, '2026-03-17 11:20:00', 'completed', 599.00),
(6, 5, '2026-02-28 16:45:00', 'completed', 19.99),
(7, 6, '2026-03-15 08:00:00', 'pending', 25.00),
(8, 7, '2026-03-17 13:10:00', 'pending', 35.50),
(9, 8, '2024-01-12 17:55:00', 'completed', 299.00),
(10, 9, '2026-03-17 12:00:00', 'pending', 29.99);

-- ORDER_ITEMS
CREATE TABLE order_items (id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, unit_price DECIMAL(10, 2));
INSERT INTO order_items VALUES
(1, 1, 1, 1, 1299.99),
(2, 2, 4, 1, 49.99),
(3, 3, 2, 1, 799.50),
(4, 4, 5, 1, 39.99),
(5, 5, 6, 1, 599.00),
(6, 6, 3, 1, 19.99),
(7, 7, 7, 1, 25.00),
(8, 8, 8, 1, 35.50),
(9, 9, 9, 1, 299.00),
(10, 10, 10, 1, 29.99);

-- REVIEWS
CREATE TABLE reviews (id INT PRIMARY KEY, product_id INT, user_id INT, rating INT, comment TEXT, created_at TIMESTAMP);
INSERT INTO reviews VALUES
(1, 1, 2, 5, 'Amazing laptop!', '2026-03-15 10:00:00'),
(2, 2, 3, 1, 'Battery drains too fast.', '2026-02-20 14:30:00'),
(3, 3, 4, 4, 'Good quality cotton.', '2026-03-10 09:15:00'),
(4, 1, 5, 5, 'Best purchase ever.', '2026-03-16 11:20:00'),
(5, 5, 6, 2, 'Size runs small.', '2026-01-20 16:45:00'),
(6, 6, 7, 4, 'Great value for money.', '2026-02-28 08:00:00'),
(7, 7, 8, 5, 'Sturdy and reliable.', '2026-03-05 13:10:00'),
(8, 8, 9, 1, 'Arrived broken.', '2026-03-12 17:55:00'),
(9, 9, 10, 3, 'Its okay for the price.', '2026-01-05 12:00:00'),
(10, 10, 1, 5, 'Fits perfectly.', '2026-03-17 09:00:00');

-- SHIPPING_DETAILS
CREATE TABLE shipping_details (id INT PRIMARY KEY, order_id INT, address_line1 VARCHAR(255), city VARCHAR(100), state VARCHAR(100), postal_code VARCHAR(20), tracking_number VARCHAR(100), carrier VARCHAR(50));
INSERT INTO shipping_details VALUES
(1, 1, '123 Main St', 'Bogota', 'Cundinamarca', '110111', 'TRK-001', 'FedEx'),
(2, 2, '456 Elm St', 'Medellin', 'Antioquia', '050001', 'TRK-002', 'UPS'),
(3, 3, '789 Oak Ave', 'Cali', 'Valle', '760001', 'TRK-003', 'FedEx'),
(4, 4, '321 Pine Rd', 'Cartagena', 'Bolivar', '130001', 'TRK-004', 'DHL'),
(5, 5, '654 Cedar Ln', 'Barranquilla', 'Atlantico', '080001', 'TRK-005', 'FedEx'),
(6, 6, '987 Birch Blvd', 'Bucaramanga', 'Santander', '680001', 'TRK-006', 'UPS'),
(7, 7, '147 Walnut St', 'Pereira', 'Risaralda', '660001', 'TRK-007', 'FedEx'),
(8, 8, '258 Cherry Ct', 'Manizales', 'Caldas', '170001', 'TRK-008', 'DHL'),
(9, 9, '369 Spruce Way', 'Santa Marta', 'Magdalena', '470001', 'TRK-009', 'FedEx'),
(10, 10, '741 Ash Dr', 'Cucuta', 'Norte de Santander', '540001', 'TRK-010', 'UPS');

-- PROMOTIONS
CREATE TABLE promotions (id INT PRIMARY KEY, code VARCHAR(50) UNIQUE, discount_percentage DECIMAL(5, 2), start_date DATE, end_date DATE, is_active BOOLEAN);
INSERT INTO promotions VALUES
(1, 'VERANO20', 20.00, '2026-03-01', '2026-03-31', TRUE),
(2, 'WELCOME10', 10.00, '2026-01-01', '2026-12-31', TRUE),
(3, 'FLASH50', 50.00, '2026-03-15', '2026-03-20', TRUE),
(4, 'WINTER30', 30.00, '2025-12-01', '2026-02-28', FALSE),
(5, 'VIP25', 25.00, '2026-01-01', '2026-12-31', TRUE),
(6, 'SPRING15', 15.00, '2026-03-20', '2026-06-20', FALSE),
(7, 'BF2025', 40.00, '2025-11-25', '2025-11-30', FALSE),
(8, 'CYBER2025', 45.00, '2025-12-01', '2025-12-02', FALSE),
(9, 'FREESHIP', 100.00, '2026-03-01', '2026-03-31', TRUE),
(10, 'HALLOWEEN', 15.00, '2025-10-25', '2025-10-31', FALSE);
"""
conn.execute(setup_sql)
print("Base cargada")

Base cargada


In [42]:
conn.execute("SHOW TABLES").df()

,name
0,categories
1,order_items
2,orders
3,products
4,promotions
5,reviews
6,shipping_details
7,users


In [43]:
result_df = conn.execute(sql_query).df()
result_df

,count_star()
0,2


In [44]:
def answer_question(question: str):
    retrieval_output = retrieve_relevant_tables(
        question=question,
        retriever=retriever,
        table_lookup=table_lookup,
        k=3
    )

    sql_query = generate_sql(
        question=retrieval_output["question"],
        retrieved_context=retrieval_output["context"]
    )

    result_df = conn.execute(sql_query).df()

    return {
        "question": question,
        "selected_tables": retrieval_output["selected_tables"],
        "context": retrieval_output["context"],
        "sql_query": sql_query,
        "result_df": result_df
    }

In [45]:
response = answer_question("¿Cuántos pedidos cancelados hubo en 2024?")

print("Pregunta:")
print(response["question"])

print("\nTablas seleccionadas:")
print(response["selected_tables"])

print("\nSQL generado:")
print(response["sql_query"])

print("\nResultado:")
response["result_df"]

Pregunta:
¿Cuántos pedidos cancelados hubo en 2024?

Tablas seleccionadas:
['orders', 'shipping_details', 'order_items']

SQL generado:
SELECT COUNT(*) FROM orders WHERE status = 'cancelled' AND EXTRACT(YEAR FROM order_date) = 2024;

Resultado:


,count_star()
0,2


In [49]:
response = answer_question_safe("Muéstrame los productos inactivos o sin stock")

print("Pregunta:")
print(response["question"])

print("\nTablas seleccionadas:")
print(response["selected_tables"])

print("\nSQL generado:")
print(response["sql_query"])

print("\nError:")
print(response["error"])

print("\nResultado:")
response["result_df"]

Pregunta:
Muéstrame los productos inactivos o sin stock

Tablas seleccionadas:
['products', 'order_items', 'promotions']

SQL generado:
SELECT name, sku FROM products WHERE stock_quantity = 0 OR is_active = FALSE;

Error:
None

Resultado:


,name,sku
0,Smartphone X,SKU-002
1,Summer Dress,SKU-005
2,Table Lamp,SKU-008
3,Budget Phone,SKU-009


In [50]:
def format_result_for_chat(response: dict) -> str:
    if response["error"]:
        return f"Hubo un error al ejecutar la consulta:\n{response['error']}"

    df = response["result_df"]

    if df is None or df.empty:
        return "No se encontraron resultados."

    return df.to_markdown(index=False)

In [51]:
response = answer_question_safe("Muéstrame los productos inactivos o sin stock")

print("SQL:")
print(response["sql_query"])

print("\nRespuesta para chat:")
print(format_result_for_chat(response))

SQL:
SELECT name, sku FROM products WHERE stock_quantity = 0 OR is_active = FALSE;

Respuesta para chat:
| name         | sku     |
|:-------------|:--------|
| Smartphone X | SKU-002 |
| Summer Dress | SKU-005 |
| Table Lamp   | SKU-008 |
| Budget Phone | SKU-009 |


In [52]:
def chatbot_response(question: str) -> dict:
    response = answer_question_safe(question)

    return {
        "question": response["question"],
        "tables": response["selected_tables"],
        "sql": response["sql_query"],
        "answer": format_result_for_chat(response),
        "error": response["error"]
    }

In [55]:
chat_response = chatbot_response("¿Cuántos pedidos cancelados hubo en 2024?")

print("Pregunta:")
print(chat_response["question"])

print("\nTablas:")
print(chat_response["tables"])

print("\nSQL:")
print(chat_response["sql"])

print("\nError:")
print(chat_response["error"])

print("\nRespuesta:")
print(chat_response["answer"])

Pregunta:
¿Cuántos pedidos cancelados hubo en 2024?

Tablas:
['orders', 'shipping_details', 'order_items']

SQL:
SELECT COUNT(*) AS total_cancelled_orders FROM orders WHERE status = 'cancelled' AND EXTRACT(YEAR FROM order_date) = 2024;

Error:
None

Respuesta:
|   total_cancelled_orders |
|-------------------------:|
|                        2 |
